# Day 40 — Final Project: Build Your Portfolio Piece
**Estimated time:** 3–4 hours  
**Requirements:**
- At least 5 SQL window functions
- At least 2 CTEs
- pandas merges (at least one non-trivial join)
- A reusable function library (at least 3 functions)
- 3 charts (publication-quality)
- Output: a polished notebook you could share on GitHub or show in an interview

---
> **Goal:** This is the notebook you'd show a analytics hiring manager during the interview when they ask "can you show me something you've built?" Every cell should be something you're proud of. No placeholder comments, no unrun cells, no error outputs.

> **Portfolio note:** When you're done, re-run from the top to verify the entire notebook runs clean. Fix every error. A notebook that doesn't run is a red flag in an interview.


## 0. Project Setup and Function Library

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  ANALYTICS PORTFOLIO PROJECT
#  Author: [Your Name]
#  Date: April 2026
#  Dataset: SAP-style analytics datasets (materials, sales, cost centers, HR, BW KPIs)
# ═══════════════════════════════════════════════════════════════════════════

import sqlite3, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
from pathlib import Path
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#fafafa',
                     'axes.grid': True, 'grid.alpha': 0.3, 'axes.spines.top': False,
                     'axes.spines.right': False})

DATA = Path('../datasets')
TODAY = pd.Timestamp('2026-04-01')

# ─── REUSABLE FUNCTION LIBRARY ───────────────────────────────────────────────

def load_all_datasets(data_path: Path = DATA) -> dict[str, pd.DataFrame]:
    """Load all 5 source datasets with correct dtypes."""
    return {
        'materials':  pd.read_csv(data_path/'materials_inventory.csv',
                                  parse_dates=['LAST_MOVEMENT_DATE']),
        'sales':      pd.read_csv(data_path/'sales_orders.csv',
                                  parse_dates=['ERDAT']),
        'cost_centers': pd.read_csv(data_path/'cost_center_actuals.csv'),
        'hr':         pd.read_csv(data_path/'hr_headcount.csv',
                                  parse_dates=['HIRE_DATE','TERM_DATE']),
        'bw_kpis':    pd.read_csv(data_path/'bw_sales_kpis.csv'),
    }


def build_sqlite_db(dfs: dict[str, pd.DataFrame]) -> sqlite3.Connection:
    """Load DataFrames into an in-memory SQLite database."""
    con = sqlite3.connect(':memory:')
    for name, df in dfs.items():
        df.to_sql(name, con, if_exists='replace', index=False)
    return con


def fmt_millions(x, _): return f'${x/1e6:.1f}M'
def fmt_thousands(x, _): return f'${x/1e3:.0f}K'
def fmt_pct(x, _): return f'{x:.1f}%'


dfs = load_all_datasets()
con = build_sqlite_db(dfs)
q = lambda sql: pd.read_sql_query(sql, con)

print('Portfolio project loaded. Datasets:', {k: len(v) for k,v in dfs.items()})

## Section 1 — Executive Performance Summary (SQL with 2 CTEs + 3 Window Functions)

In [ ]:
# YOUR SOLUTION
# Requirements: 2 CTEs, at least 3 window functions

executive_sql = '''
-- CTE 1: Monthly revenue with YoY comparison using LAG (window function #1)
WITH monthly_revenue AS (
    SELECT
        CAL_YEAR_MONTH,
        CAL_YEAR_MONTH / 100 AS yr,
        CAL_YEAR_MONTH % 100 AS mo,
        REGION,
        ROUND(SUM(REVENUE), 0) AS revenue,
        ROUND(SUM(GROSS_MARGIN), 0) AS gross_margin,
        ROUND(SUM(DISCOUNT_AMT), 0) AS discounts
    FROM bw_kpis
    GROUP BY CAL_YEAR_MONTH, REGION
),

-- CTE 2: YoY and running totals using window functions #2, #3, #4, #5
revenue_analysis AS (
    SELECT
        cy.CAL_YEAR_MONTH,
        cy.yr,
        cy.mo,
        cy.REGION,
        cy.revenue                                              AS cy_revenue,
        py.revenue                                              AS py_revenue,
        ROUND((cy.revenue - py.revenue) /
              NULLIF(py.revenue, 0) * 100, 1)                  AS yoy_pct,

        -- Window function #2: running total within year and region
        SUM(cy.revenue) OVER (
            PARTITION BY cy.yr, cy.REGION
            ORDER BY cy.mo
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )                                                       AS ytd_revenue,

        -- Window function #3: rank region by revenue within each month
        RANK() OVER (
            PARTITION BY cy.CAL_YEAR_MONTH
            ORDER BY cy.revenue DESC
        )                                                       AS region_rank,

        -- Window function #4: 3-month rolling average
        AVG(cy.revenue) OVER (
            PARTITION BY cy.REGION
            ORDER BY cy.CAL_YEAR_MONTH
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        )                                                       AS rolling_3m_avg,

        -- Window function #5: revenue share of month total
        ROUND(cy.revenue /
              SUM(cy.revenue) OVER (PARTITION BY cy.CAL_YEAR_MONTH) * 100,
              1)                                                AS region_share_pct
    FROM monthly_revenue cy
    LEFT JOIN monthly_revenue py
        ON cy.REGION = py.REGION
        AND cy.mo = py.mo
        AND cy.yr = py.yr + 1
)

SELECT *
FROM revenue_analysis
WHERE yr = 2025
ORDER BY CAL_YEAR_MONTH, region_rank
'''

perf_df = q(executive_sql)
print(f'Executive analysis: {len(perf_df)} rows across {perf_df["REGION"].nunique()} regions')
print(perf_df.head(12).to_string(index=False))

## Section 2 — Inventory Risk Analysis (pandas merges + derived metrics)

In [ ]:
# YOUR SOLUTION
# Requirements: at least one non-trivial pandas merge

mat = dfs['materials'].copy()
sales = dfs['sales'].copy()

# Calculate days since movement
mat['DAYS_STALE'] = (TODAY - mat['LAST_MOVEMENT_DATE']).dt.days.fillna(999).astype(int)
mat['INV_VALUE'] = mat['LABST'] * mat['STPRS']

# Aging buckets
bins = [0, 30, 90, 180, 9999]
labels = ['Current', 'Slow', 'Stagnant', 'Dead']
mat['BUCKET'] = pd.cut(mat['DAYS_STALE'], bins=bins, labels=labels)

# Sales velocity (last 6 months of delivered orders)
recent_sales = sales[
    (sales['STATUS'].isin(['DELIVERED','CLOSED'])) &
    (sales['ERDAT'] >= TODAY - pd.DateOffset(months=6))
]
velocity = (recent_sales.groupby('MATNR')['MENGE'].sum() / 6).reset_index()
velocity.columns = ['MATNR', 'MONTHLY_VEL']

# Merge 1: inventory + velocity
mat_v = mat.merge(velocity, on='MATNR', how='left')
mat_v['MONTHLY_VEL'] = mat_v['MONTHLY_VEL'].fillna(0)
mat_v['MONTHS_COVER'] = np.where(
    mat_v['MONTHLY_VEL'] > 0,
    mat_v['LABST'] / mat_v['MONTHLY_VEL'],
    np.inf
)
mat_v['EXCESS'] = mat_v['MONTHS_COVER'] > 3

# Merge 2: inventory + open orders (count of open orders per MATNR)
open_demand = (
    sales[sales['STATUS']=='OPEN']
    .groupby('MATNR').agg(OPEN_ORDERS=('VBELN','count'), OPEN_VALUE=('NETWR','sum'))
    .reset_index()
)
mat_full = mat_v.merge(open_demand, on='MATNR', how='left')
mat_full['OPEN_ORDERS'] = mat_full['OPEN_ORDERS'].fillna(0)

# Risk score: dead + no open demand
mat_full['HIGH_RISK'] = (mat_full['BUCKET'] == 'Dead') & (mat_full['OPEN_ORDERS'] == 0)

summary = mat_full.groupby('BUCKET', observed=True).agg(
    ITEMS=('MATNR','count'),
    INV_VALUE=('INV_VALUE','sum'),
    EXCESS_COUNT=('EXCESS','sum'),
    HIGH_RISK_COUNT=('HIGH_RISK','sum')
).reset_index()

print('=== Inventory Risk Summary ===')
print(summary.to_string(index=False))
print(f'\nHigh-risk materials (dead + no demand): {mat_full["HIGH_RISK"].sum()}')
print(f'High-risk value: ${mat_full.loc[mat_full["HIGH_RISK"],"INV_VALUE"].sum():,.0f}')

## Section 3 — Workforce Cost Analysis

In [ ]:
# YOUR SOLUTION
hr = dfs['hr'].copy()
cc = dfs['cost_centers'].copy()

active = hr[hr['TERM_DATE'].isna()].copy()
active['TENURE_YRS'] = (TODAY - active['HIRE_DATE']).dt.days / 365.25

# Salary by org and tenure band
active['TENURE_BAND'] = pd.cut(
    active['TENURE_YRS'],
    bins=[0,1,3,5,10,100],
    labels=['<1yr','1-3yr','3-5yr','5-10yr','10+yr']
)

org_summary = (
    active.groupby('ORGTX')
    .agg(HC=('PERNR','count'), TOTAL_SAL=('SALARY','sum'), AVG_SAL=('SALARY','mean'))
    .reset_index()
    .sort_values('TOTAL_SAL', ascending=False)
)

# Cost center budget vs actual salary (using KSTAR=400000 = Salaries)
cc_salary = (
    cc[(cc['KSTAR']==400000) & (cc['GJAHR']==2024)]
    .groupby('KOSTL')
    .agg(YTD_ACTUAL=('ACTUAL_AMT','sum'), YTD_PLAN=('PLAN_AMT','sum'))
    .reset_index()
)
hr_cost = (
    active.groupby('KOSTL')['SALARY'].sum()
    .reset_index()
    .rename(columns={'SALARY':'ANNUAL_HC_COST'})
)
cost_compare = hr_cost.merge(cc_salary, on='KOSTL', how='inner')
cost_compare['OVERAGE'] = cost_compare['ANNUAL_HC_COST'] - cost_compare['YTD_PLAN']
cost_compare['VAR_PCT'] = (cost_compare['OVERAGE'] / cost_compare['YTD_PLAN'] * 100).round(1)

print('=== Workforce Cost vs Plan (Top 8 Over-Budget) ===')
print(cost_compare.nlargest(8, 'OVERAGE').to_string(index=False))

## Section 4 — Three Publication-Quality Charts

In [ ]:
# YOUR SOLUTION — 3 charts required

fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)

# ── Chart 1: YoY Revenue by Region (grouped bars) ────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
bw = dfs['bw_kpis']

rev_yoy = (
    bw[bw['CAL_YEAR_MONTH']//100.isin([2024,2025])]
    .groupby([bw['CAL_YEAR_MONTH']//100, 'REGION'])['REVENUE']
    .sum()
    .unstack('REGION')
    .T
)
if 2024 in rev_yoy.columns and 2025 in rev_yoy.columns:
    x = np.arange(len(rev_yoy))
    w = 0.38
    bars1 = ax1.bar(x - w/2, rev_yoy[2024]/1e6, w, label='2024', color='#95a5a6', alpha=0.8)
    bars2 = ax1.bar(x + w/2, rev_yoy[2025]/1e6, w, label='2025', color='#2980b9', alpha=0.9)
    ax1.set_xticks(x)
    ax1.set_xticklabels(rev_yoy.index, fontsize=9)
    ax1.set_ylabel('Revenue', fontsize=9)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
    ax1.set_title('Revenue by Region: 2024 vs 2025', fontsize=11, fontweight='bold', pad=10)
    ax1.legend(fontsize=9)
    for bar in bars2:
        h = bar.get_height()
        ax1.text(bar.get_x()+bar.get_width()/2, h+0.02, f'${h:.1f}M', ha='center', fontsize=7, color='#2c3e50')

# ── Chart 2: Inventory Value by Bucket (horizontal bar, colour-coded risk) ───
ax2 = fig.add_subplot(gs[0, 2])
bucket_val = mat_full.groupby('BUCKET', observed=True)['INV_VALUE'].sum() / 1e3
c_map = {'Current':'#2ecc71','Slow':'#f39c12','Stagnant':'#e67e22','Dead':'#e74c3c'}
bucket_val.plot(kind='barh', ax=ax2,
                color=[c_map.get(str(b),'#95a5a6') for b in bucket_val.index],
                edgecolor='white')
ax2.set_title('Inventory at Risk ($K)', fontsize=11, fontweight='bold', pad=10)
ax2.set_xlabel('Value ($K)', fontsize=9)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_thousands))
ax2.invert_yaxis()
for i, (v, label) in enumerate(zip(bucket_val.values, bucket_val.index)):
    ax2.text(v + 5, i, f'${v:,.0f}K', va='center', fontsize=8)

# ── Chart 3: Workforce Cost vs Plan (waterfall-style diverging bar) ───────────
ax3 = fig.add_subplot(gs[1, :])
top_cc = cost_compare.nlargest(12, 'OVERAGE').sort_values('OVERAGE')
colors_cc = ['#e74c3c' if v >= 0 else '#2ecc71' for v in top_cc['OVERAGE']]
ax3.barh(range(len(top_cc)), top_cc['OVERAGE']/1e3, color=colors_cc, edgecolor='white', alpha=0.85)
ax3.set_yticks(range(len(top_cc)))
ax3.set_yticklabels(top_cc['KOSTL'], fontsize=8)
ax3.axvline(0, color='black', linewidth=1)
ax3.set_title('Headcount Cost vs Salary Budget by Cost Center ($K)',
              fontsize=11, fontweight='bold', pad=10)
ax3.set_xlabel('Overage ($K)', fontsize=9)
ax3.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_thousands))

# Main title
fig.suptitle('SAP Analytics Portfolio — Business Review April 2026',
             fontsize=14, fontweight='bold', y=0.98, color='#2c3e50')

plt.savefig('day40_portfolio.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Portfolio charts saved.')

## Section 5 — Narrative Summary and Self-Reflection

In [ ]:
# YOUR SOLUTION — write a professional narrative

cy_rev_total = bw[bw['CAL_YEAR_MONTH']//100==2025]['REVENUE'].sum()
py_rev_total = bw[bw['CAL_YEAR_MONTH']//100==2024]['REVENUE'].sum()
yoy = (cy_rev_total - py_rev_total) / py_rev_total * 100
gm_pct = bw[bw['CAL_YEAR_MONTH']//100==2025]['GROSS_MARGIN'].sum() / cy_rev_total * 100
dead_exposure = mat_full.loc[mat_full['BUCKET']=='Dead','INV_VALUE'].sum()
high_risk_items = mat_full['HIGH_RISK'].sum()

narrative = f"""
╔══════════════════════════════════════════════════════════════════════╗
║         PORTFOLIO PROJECT — EXECUTIVE NARRATIVE SUMMARY             ║
╠══════════════════════════════════════════════════════════════════════╣

The business delivered ${cy_rev_total/1e6:.1f}M in revenue for 2025, a {yoy:+.1f}% improvement 
versus 2024, driven by sustained momentum in the Central and East regions.

Gross margin of {gm_pct:.1f}% remains under pressure from a discount rate of 
{bw[bw['CAL_YEAR_MONTH']//100==2025]['DISCOUNT_AMT'].sum()/cy_rev_total*100:.1f}% — a 1pp reduction
would recover approximately ${cy_rev_total*0.01/1e3:.0f}K of gross profit annually.

Inventory health requires immediate attention: ${dead_exposure/1e3:.0f}K of stock 
(>180 days no movement) with no open demand represents {high_risk_items} materials 
at risk of write-down in the Q2 close cycle.

Workforce costs are broadly in line with plan, with 
{(cost_compare['OVERAGE']>0).sum()} cost centers showing salary spend above 
their CO budget — primarily in Operations and Finance org units.

Next actions: (1) Initiate slow-moving stock review with supply chain before 
period 6 close; (2) Review discount approval workflow to reduce effective 
discount rate; (3) Validate FY2026 budget submissions against updated headcount 
actual data.

╚══════════════════════════════════════════════════════════════════════╝
"""
print(narrative)

## Completeness Checklist
Before sharing this notebook, verify:
- [ ] ≥ 5 SQL window functions used (ROW_NUMBER, RANK, LAG, SUM OVER, AVG OVER, etc.)
- [ ] ≥ 2 CTEs written
- [ ] At least 2 non-trivial pandas merges
- [ ] At least 3 reusable functions in the function library
- [ ] 3 charts, all with titles, labelled axes, and formatted tick marks
- [ ] Entire notebook runs clean (Kernel → Restart & Run All)
- [ ] No placeholder comments or `# YOUR CODE HERE` remaining

## Final Reflection
**Answer in a new markdown cell below:**

> "What would you add if you had 2 more hours?"

Prompt: Write 3–5 specific enhancements you'd make — not generic ("more charts") but specific ("I'd add a 12-month revenue forecast using a simple linear regression on the monthly trend, and overlay it on Chart 1 with a confidence interval").

This question appears in real enterprise analytics interviews. A specific, data-literate answer distinguishes a strong candidate.
